<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/Phase1_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1 — SQL Analytics Layer (Business Intelligence Foundation)
## DB creation, setup and csv upload
### Using sqlite to create database

In [39]:
import sqlite3
conn = sqlite3.connect('capstone.db')
cursor = conn.cursor()
print("Connected to sqlite database 'capstone.db'")

Connected to sqlite database 'capstone.db'


### Create Tables
#### Since sqlite is being used as database.
#### Tables will be created with `STRICT` keyword
#### This guarantees that datatype checking performed and incorrect datatype is prevented from uploading.
#### Creating 'patients' table

In [40]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS patients (
    patient_id INTEGER PRIMARY KEY,
    age INTEGER,
    gender TEXT,
    city TEXT,
    insurance_provider TEXT,
    chronic_flag INTEGER,
    registration_date TEXT
) STRICT; -- This will block 'wrong' data types entirely
''')

#### Creating 'visits' table

In [41]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS visits (
    visit_id INTEGER PRIMARY KEY,
    patient_id INTEGER,
    visit_date TEXT,
    department TEXT,
    visit_type TEXT,
    length_of_stay_hours REAL,
    risk_score TEXT,
    doctor_id INTEGER,
    FOREIGN KEY (patient_id) REFERENCES patients (patient_id)
) STRICT; -- This will block 'wrong' data types entirely
''')


#### Creating 'billing' table

In [42]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS billing (
    bill_id INTEGER PRIMARY KEY,
    visit_id INTEGER,
    billed_amount REAL,
    approved_amount REAL,
    claim_status TEXT,
    payment_days REAL,
    billing_date TEXT,
    FOREIGN KEY (visit_id) REFERENCES visits (visit_id)
) STRICT; -- This will block 'wrong' data types entirely
''')


In [43]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(tables)

[('patients',), ('visits',), ('billing',)]


#### Populate tables with csv files

In [44]:
# Upload file to google.colab
import io
from google.colab import files, drive
import pandas as pd

drive.mount('/content/drive')

# comment this code, if files already uploaded.
# uploaded = files.upload()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
#IMPORTANT!!! replace the file path according to your location respectively.
# Populate patients table
patients_df = pd.read_csv('/content/drive/MyDrive/files/capstone/patients.csv')
# Clear existing data
cursor.execute('DELETE FROM patients')
patients_df.to_sql('patients', conn, if_exists='append', index=False)
print("Patients data uploaded successfully.")

# Populate visits table
visits_df = pd.read_csv('/content/drive/MyDrive/files/capstone/visits.csv')
# Clear existing data
cursor.execute('DELETE FROM visits')
visits_df.to_sql('visits', conn, if_exists='append', index=False)
print("Visits data uploaded successfully.")

# Populate billing table
billing_df = pd.read_csv('/content/drive/MyDrive/files/capstone/billing.csv')
# Clear existing data
cursor.execute('DELETE FROM billing')
billing_df.to_sql('billing', conn, if_exists='append', index=False)
print("Billing data uploaded successfully.")

conn.commit()
print("All data uploaded and committed successfully.")

# Verify data count
print("\nVerifying row counts:")
cursor.execute("SELECT COUNT(*) FROM patients")
print(f"Patients table has {cursor.fetchone()[0]} rows.")
cursor.execute("SELECT COUNT(*) FROM visits")
print(f"Visits table has {cursor.fetchone()[0]} rows.")
cursor.execute("SELECT COUNT(*) FROM billing")
print(f"Billing table has {cursor.fetchone()[0]} rows.")

Patients data uploaded successfully.
Visits data uploaded successfully.
Billing data uploaded successfully.
All data uploaded and committed successfully.

Verifying row counts:
Patients table has 5000 rows.
Visits table has 25000 rows.
Billing table has 25000 rows.


In [46]:
# verify table data, fetching first 10 rows of billing table
cursor.execute('SELECT * FROM billing LIMIT 10')

# Get column names
column_names = [description[0] for description in cursor.description]

# Get data rows
rows = cursor.fetchall()

df_billing = pd.DataFrame(rows, columns=column_names)
print("\n--- Verifying table data: Billing Table (First 10 Rows) ---\n")
display(df_billing)



--- Verifying table data: Billing Table (First 10 Rows) ---



,bill_id,visit_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,1,1,23577.37,0.00,Rejected,16.0,2025-06-18
1,2,2,38178.81,38178.81,Paid,18.0,2025-10-09
2,3,3,5038.97,5038.97,Paid,NaN,2025-01-20
3,4,4,22813.34,22813.34,Paid,16.0,2025-12-24
4,5,5,27106.95,27106.95,Paid,14.0,2025-09-23
5,6,6,19453.77,19453.77,Paid,17.0,2026-01-12
6,7,7,6269.47,4437.65,Pending,10.0,2025-11-16
7,8,8,38178.73,38178.73,Paid,6.0,2025-06-27
8,9,9,21258.56,21258.56,Paid,15.0,2025-10-06
9,10,10,18161.42,18161.42,Paid,13.0,2025-07-16


## Operational Analysis
1. Retrieve the top 10 departments by total visit volume.
2. Identify the top 5 departments with the highest average length of stay.
3. Find the percentage of High Risk visits per department.
4. Determine the average number of visits per patient by city.
5. Identify doctors handling the highest number of High Risk visits.





In [47]:
#1. Retrieve the top 10 departments by total visit volume.
cursor.execute('''
SELECT department, COUNT(*) AS visit_count
FROM visits
GROUP BY department
ORDER BY visit_count
DESC LIMIT 10
''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_top_departments_by_visit_volume = pd.DataFrame(result, columns=column_names)
print("\n--- Operational Analysis: Top 10 departments by total visit volume ---\n")
display(df_top_departments_by_visit_volume)




--- Operational Analysis: Top 10 departments by total visit volume ---



,department,visit_count
0,General,4228
1,ER,4220
2,Neurology,4165
3,Orthopedics,4164
4,Cardiology,4159
5,ICU,4064


In [48]:
#2. Identify the top 5 departments with the highest average length of stay.
cursor.execute('''
  SELECT DEPARTMENT, AVG(LENGTH_OF_STAY_HOURS) AS AVG_LENGTH_OF_STAY_HOURS
  FROM VISITS
  GROUP BY DEPARTMENT
  ORDER BY AVG_LENGTH_OF_STAY_HOURS DESC
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_top_5_depts_with_avg_stay_hours = pd.DataFrame(result, columns=column_names)
print("\n--- Operational Analysis: Top 5 departments with the highest average length of stay ---\n")
display(df_top_5_depts_with_avg_stay_hours)



--- Operational Analysis: Top 5 departments with the highest average length of stay ---



,department,AVG_LENGTH_OF_STAY_HOURS
0,Neurology,19.718098
1,Orthopedics,19.662656
2,Cardiology,19.600962
3,ER,19.534967
4,General,19.434905
5,ICU,19.355234


In [49]:
#3. Find the percentage of High Risk visits per department.
cursor.execute('''
  select department,
  count(*) as total_visits,
  sum(case when risk_score ='High' Then 1 else 0 END) as high_risk_visits,
  round( sum(case when risk_score='High' then 1 else 0 end) * 100 / count(*), 2) as high_risk_percentage
  from visits
  group by department;
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_percentage_of_high_risk_visit_per_dept = pd.DataFrame(result, columns=column_names)
print("\n--- Operational Analysis: Percentage of High Risk visits per department ---\n")
display(df_percentage_of_high_risk_visit_per_dept)



--- Operational Analysis: Percentage of High Risk visits per department ---



,department,total_visits,high_risk_visits,high_risk_percentage
0,Cardiology,4159,790,18.0
1,ER,4220,872,20.0
2,General,4228,839,19.0
3,ICU,4064,845,20.0
4,Neurology,4165,846,20.0
5,Orthopedics,4164,842,20.0


In [50]:
#4. Determine the average number of visits per patient by city.
cursor.execute('''
  select p.city,
  Round(count(v.visit_id) * 1.0/count(distinct p.patient_id),2) as avg_visits_per_patient
  from visits v
  join patients p on v.patient_id = p.patient_id
  group by p.city
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_avg_no_visits_per_patient = pd.DataFrame(result, columns=column_names)
print("\n--- Operational Analysis: Average number of visits per patient by city ---\n")
display(df_avg_no_visits_per_patient)



--- Operational Analysis: Average number of visits per patient by city ---



,city,avg_visits_per_patient
0,Bangalore,5.02
1,Chennai,5.02
2,Delhi,4.95
3,Hyderabad,5.06
4,Mumbai,5.02
5,Pune,5.12


In [51]:
#5. Identify doctors handling the highest number of High Risk visits.
# As the no. of doctors(or rows)  are not defined as such only top 10 doctors are displayed
cursor.execute('''
  select doctor_id , count(risk_score) as high_risk_count
  from visits where risk_score = 'High'
  group by  doctor_id
  order by high_risk_count desc Limit 10;
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_doc_with_highest_high_risk_visists = pd.DataFrame(result, columns=column_names)
print("\n--- Operational Analysis: Top 10 Doctor's Id handling the highest number of High Risk visits ---\n")
display(df_doc_with_highest_high_risk_visists)



--- Operational Analysis: Top 10 Doctor's Id handling the highest number of High Risk visits ---



,doctor_id,high_risk_count
0,174,71
1,198,69
2,169,68
3,177,67
4,135,65
5,105,65
6,188,64
7,180,64
8,131,62
9,178,61


## Financial Analysis
1. Retrieve the top 10 insurance providers by total billed amount.
2. Identify the top 5 insurance providers with the highest claim rejection rate.
3. Find the average payment delay (payment_days) by insurance provider.
4. Calculate the revenue realization ratio (approved_amount / billed_amount) by department.
5. Identify visits where billed_amount is high but approved_amount is zero or missing.



In [52]:

#1. Retrieve the top 10 insurance providers by total billed amount.
cursor.execute('''
select distinct(p.insurance_provider),
printf("%,d", sum(b.billed_amount)) as total_billed_amount
from patients p
join visits v on p.patient_id = v.patient_id
join billing b on v.visit_id = b.visit_id
group by p.insurance_provider
order by total_billed_amount desc
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_top_10_insurance_provider = pd.DataFrame(result, columns=column_names)
print('\n--- Financial Analysis: Top 10 Insurance provider by total billed amount ---\n')
display(df_top_10_insurance_provider)



--- Financial Analysis: Top 10 Insurance provider by total billed amount ---



,insurance_provider,total_billed_amount
0,MediCareX,"134,591,163"
1,CareOne,"130,707,992"
2,HealthPlus,"130,180,740"
3,SecureLife,"126,289,039"


In [53]:
#2. Identify the top 5 insurance providers with the highest claim rejection rate.
cursor.execute('''
select distinct(p.insurance_provider),
count(b.billed_amount) as total_rejection_count
from patients p
join visits v on p.patient_id = v.patient_id
join billing b on v.visit_id = b.visit_id
where b.claim_status = 'Rejected'
group by p.insurance_provider
order by total_rejection_count desc
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_top_5_insurance_provider_by_claim_rejection_rate = pd.DataFrame(result, columns=column_names)
print('\n--- Financial Analysis: Top 5 Insurance providers with the highest claim rejection rate ---\n')
display(df_top_5_insurance_provider_by_claim_rejection_rate)





--- Financial Analysis: Top 5 Insurance providers with the highest claim rejection rate ---



,insurance_provider,total_rejection_count
0,MediCareX,996
1,SecureLife,936
2,CareOne,934
3,HealthPlus,931


In [54]:
#3. Find the average payment delay (payment_days) by insurance provider.
cursor.execute('''
select distinct(p.insurance_provider),
round(avg(b.payment_days),2) as "avg_payment_delay(days)"
from patients p
join visits v on p.patient_id = v.patient_id
join billing b on v.visit_id = b.visit_id
group by p.insurance_provider
order by avg(b.payment_days) desc
''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_avg_payment_delay_by_insurance_provider = pd.DataFrame(result, columns=column_names)
print('\n--- Financial Analysis: Average payment delay (payment_days) by insurance provider ---\n')
display(df_avg_payment_delay_by_insurance_provider)



--- Financial Analysis: Average payment delay (payment_days) by insurance provider ---



,insurance_provider,avg_payment_delay(days)
0,HealthPlus,13.08
1,SecureLife,13.08
2,CareOne,13.03
3,MediCareX,13.01


In [55]:
#4. Calculate the revenue realization ratio (approved_amount / billed_amount) by department.
cursor.execute('''
select v.department,
sum(b.approved_amount) as total_approved_amount,
sum(b.billed_amount) as total_billed_amount,
sum(b.approved_amount)/sum(b.billed_amount) as realization_ratio
from visits v
join billing b on v.visit_id = b.visit_id
group by v.department
order by realization_ratio desc
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_revenue_realization_ratio_by_department = pd.DataFrame(result, columns=column_names)
print('\n--- Financial Analysis: Revenue realization ratio (approved_amount / billed_amount) by department ---\n')
display(df_revenue_realization_ratio_by_department)





--- Financial Analysis: Revenue realization ratio (approved_amount / billed_amount) by department ---



,department,total_approved_amount,total_billed_amount,realization_ratio
0,ICU,63166516.84,84757763.76,0.745259
1,Orthopedics,65211585.83,87811455.80,0.742632
2,General,64690870.95,87131451.86,0.742451
3,Neurology,64708778.69,87310048.09,0.741138
4,ER,65672329.38,88686960.35,0.740496
5,Cardiology,63705806.68,86071256.19,0.740152


In [56]:
#5. Identify visits where billed_amount is high but approved_amount is zero or missing.
cursor.execute('''
select distinct(v.visit_id),
b.billed_amount,
b.approved_amount
from visits v
join billing b on v.visit_id = b.visit_id
where b.approved_amount = 0
order by b.billed_amount desc
'''
)
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_visits_where_billed_high_approved_zero = pd.DataFrame(result, columns=column_names)
print('\n--- Financial Analysis: Visits where billed_amount is high but approved_amount is zero or missing ---\n')
display(df_visits_where_billed_high_approved_zero)



--- Financial Analysis: Visits where billed_amount is high but approved_amount is zero or missing ---



,visit_id,billed_amount,approved_amount
0,18381,68213.53,0.0
1,20791,49533.30,0.0
2,8456,49059.15,0.0
3,19179,47123.69,0.0
4,2097,46529.89,0.0
...,...,...,...
3592,20822,500.00,0.0
3593,21456,500.00,0.0
3594,21932,500.00,0.0
3595,22889,500.00,0.0


## Data Quality and Integrity Checks
1. Detect visits that do not have a corresponding billing record.
2. Detect billing records that do not have a corresponding visit.
3. Identify patients with duplicate patient_id values.
4. Find records with missing or invalid length_of_stay_hours or payment_days.
5. Identify visits linked to patients with missing insurance provider information.

In [57]:
#1. Detect visits that do not have a corresponding billing record.
cursor.execute('''
select v.visit_id
from visits v
where not exists (select 1 from billing b where v.visit_id = b.visit_id)
''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_visits_without_billing = pd.DataFrame(result, columns=column_names)
print('\n--- Data Quality: Visits that do not have a corresponding billing record ---\n')
display(df_visits_without_billing)



--- Data Quality: Visits that do not have a corresponding billing record ---



,visit_id


In [58]:
#2. Detect billing records that do not have a corresponding visit.
cursor.execute('''
select b.bill_id
from billing b
where not exists (select 1 from visits v where v.visit_id = b.visit_id)
''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_billing_without_visit = pd.DataFrame(result, columns=column_names)
print('\n--- Data Quality: Billing records that do not have a corresponding visit ---\n')
display(df_billing_without_visit)




--- Data Quality: Billing records that do not have a corresponding visit ---



,bill_id


In [59]:
#3. Identify patients with duplicate patient_id values.
cursor.execute('''
select patient_id, count(*) as duplicate_count
from patients
group by patient_id
having count(*) > 1
''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_duplicate_patient_id = pd.DataFrame(result, columns=column_names)
print('\n--- Data Quality: Patients with duplicate patient_id values ---\n')
display(df_duplicate_patient_id)



--- Data Quality: Patients with duplicate patient_id values ---



,patient_id,duplicate_count


In [60]:
#4. Find records with missing or invalid length_of_stay_hours or payment_days.
cursor.execute('''
SELECT
    v.visit_id,
    v.length_of_stay_hours,
    b.payment_days,
    CASE
        WHEN v.length_of_stay_hours IS NULL OR b.payment_days IS NULL THEN 'Missing Value'
        WHEN v.length_of_stay_hours < 0 OR b.payment_days < 0 THEN 'Negative Value'
        WHEN v.length_of_stay_hours > 8760 THEN 'Outlier (Over 1 year)' -- Logic check
    END AS error_reason
FROM visits v
LEFT JOIN billing b ON v.visit_id = b.visit_id
WHERE
    -- 1. Check for Missing (NULL)
    v.length_of_stay_hours IS NULL
    OR b.payment_days IS NULL

    -- 2. Check for Invalid Logic (Negative numbers)
    OR v.length_of_stay_hours < 0
    OR b.payment_days < 0

    -- 3. Check for Outliers (previously in CASE, now part of selection criteria)
    OR v.length_of_stay_hours > 8760;
''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_missing_invalid_length_of_stay_hours_payment_days = pd.DataFrame(result, columns=column_names)
print('\n--- Data Quality: Records with missing or invalid length_of_stay_hours or payment_days ---\n')
display(df_missing_invalid_length_of_stay_hours_payment_days)



--- Data Quality: Records with missing or invalid length_of_stay_hours or payment_days ---



,visit_id,length_of_stay_hours,payment_days,error_reason
0,3,34.36,None,Missing Value
1,22,15.27,None,Missing Value
2,41,18.26,None,Missing Value
3,66,9.59,None,Missing Value
4,109,21.18,None,Missing Value
...,...,...,...,...
785,24842,10.48,None,Missing Value
786,24851,27.29,None,Missing Value
787,24896,18.86,None,Missing Value
788,24944,26.58,None,Missing Value


In [61]:
#5. Identify visits linked to patients with missing insurance provider information.
cursor.execute('''
select v.visit_id, v.patient_id, p.insurance_provider
from visits v
join patients p on v.patient_id = p.patient_id
where p.insurance_provider is null
''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_visits_linked_missing_insurance_provider = pd.DataFrame(result, columns=column_names)
print('\n--- Data Quality: Visits linked to patients with missing insurance provider information ---\n')
display(df_visits_linked_missing_insurance_provider)



--- Data Quality: Visits linked to patients with missing insurance provider information ---



,visit_id,patient_id,insurance_provider


## Summary

All tasks have been successfully completed and the notebook has been thoroughly organized and analyzed.
I've covered:

*   **Database Setup**: Tables (`patients`, `visits`, `billing`) created with `STRICT` data types and foreign key constraints.
*   **Data Population**: Data from CSV files loaded into respective tables.
*   **Operational Analysis**: Key metrics for departments, length of stay, high-risk visits, visits per patient, and doctors handling high-risk cases.
*   **Financial Analysis**: Insights into insurance providers by billed amount, claim rejection rates, payment delays, revenue realization ratios, and visits with high billed but zero approved amounts.
*   **Data Quality Checks**: Identified missing or invalid `length_of_stay_hours` or `payment_days`, and confirmed no issues with missing linked records or duplicate patient IDs.
